# CFD Analysis and Thermal Performance Optimisation of an Air-Cooled Heat Sink

Post-processing and optimisation study for a plate-fin aluminium heat sink
simulated in **ANSYS Fluent 2026 R1**.

| | |
|---|---|
| **Heat sink** | 60 x 60 x 3 mm Al-6061 base, 10 fins, 25 mm tall, 1.5 mm thick, 6.2 mm pitch |
| **Heat load** | 20 W over a central 40 x 40 mm patch (12 500 W/m²) |
| **Fluid** | Air, 300 K inlet, forced convection |
| **Turbulence** | k-omega SST |
| **Mesh** | 417 997 cells, symmetric half-model, max skewness 0.81, min orthogonal quality 0.19 |
| **Study variable** | Inlet velocity, 1 to 6 m/s |

**How to use this notebook.** Run the cells in order. Every figure cell renders
the plot, writes a 300 dpi PNG and a vector PDF, and then triggers a browser
download automatically. The final cell bundles everything into one ZIP.

## 1 · Environment and figure style

In [ ]:
# --- imports -----------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:                      # lets the notebook also run locally
    IN_COLAB = False
    class files:
        @staticmethod
        def download(path): print(f"[local] would download {path}")

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)


def export(fig, stem, dpi=300):
    """Save a figure as PNG + PDF and push both to the browser."""
    png, pdf = OUT / f"{stem}.png", OUT / f"{stem}.pdf"
    fig.savefig(png, dpi=dpi, bbox_inches="tight", facecolor=fig.get_facecolor())
    fig.savefig(pdf, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    files.download(str(png)); files.download(str(pdf))
    print(f"saved  {png}   {pdf}")

### Figure style

One style block for every figure, so the whole set reads as a single system.

The palette is a validated categorical set: the adjacent-pair separation stays
above the colour-vision-deficiency threshold, so the figures survive being
printed in greyscale or read by a colour-blind reviewer. Marks are thin, the
grid is recessive, and only the points that carry an argument get a label.

In [ ]:
# --- academic figure theme ---------------------------------------------------
PAPER   = "#ffffff"   # page behind the axes
PANEL   = "#f5f4f0"   # the plotting panel: a warm paper tint, not clinical white
GRID    = "#ffffff"   # grid drawn as light relief on the panel
FRAME   = "#d8d6cf"

INK     = "#14140f"   # headline
INK_2   = "#4a4842"   # axis labels
INK_3   = "#8a8880"   # ticks, subtitles, annotations

BLUE    = "#2a78d6"   # series 1 - thermal
ORANGE  = "#c1541c"   # series 2 - hydraulic cost
TEAL    = "#12776f"   # series 3 - combined objective
CRIMSON = "#9b2226"   # highlight / optimum marker

mpl.rcParams.update({
    "figure.figsize": (7.6, 4.8),
    "figure.dpi": 120,
    "figure.facecolor": PAPER,
    "savefig.facecolor": PAPER,
    "axes.facecolor": PANEL,

    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.labelcolor": INK_2,
    "axes.labelpad": 9,
    "axes.titlesize": 13.5,
    "axes.titlepad": 30,

    "axes.edgecolor": FRAME,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,

    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 1.1,
    "grid.alpha": 1.0,
    "axes.axisbelow": True,

    "xtick.color": INK_3, "ytick.color": INK_3,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "xtick.major.size": 0, "ytick.major.size": 0,
    "xtick.minor.size": 0, "ytick.minor.size": 0,

    "legend.frameon": True,
    "legend.facecolor": PAPER,
    "legend.edgecolor": FRAME,
    "legend.framealpha": 0.95,
    "legend.fontsize": 10,
    "legend.borderpad": 0.7,

    "lines.linewidth": 2.2,
    "lines.markersize": 7.5,
    "lines.solid_capstyle": "round",
    "mathtext.fontset": "dejavusans",
})


def titled(ax, headline, subtitle=None):
    """Left-aligned headline in a serif face, grey subtitle beneath it."""
    ax.set_title(headline, loc="left", color=INK, fontsize=13.5,
                 fontweight="bold", fontfamily="DejaVu Serif")
    if subtitle:
        ax.text(0, 1.015, subtitle, transform=ax.transAxes, ha="left", va="bottom",
                fontsize=10, color=INK_3)


def series(ax, x, y, color, label=None, marker="o", ls="-"):
    """A line with panel-ringed markers so overlapping points stay separable."""
    return ax.plot(x, y, color=color, marker=marker, linestyle=ls, label=label,
                   markerfacecolor=color, markeredgecolor=PANEL,
                   markeredgewidth=1.6, zorder=3)


def tag(ax, x, y, text, dx=8, dy=8, color=None, weight="bold", size=10):
    ax.annotate(text, (x, y), textcoords="offset points", xytext=(dx, dy),
                fontsize=size, color=color or INK_2, fontweight=weight, zorder=4)


def polish(ax):
    """Minor ticks for readability without extra grid clutter."""
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))

print("style loaded")

## 2 · Simulation results

Six converged steady-state runs. `Tmax` is the facet maximum of static
temperature on the heated patch; `dP` is the area-weighted static pressure at
the inlet with the outlet held at 0 Pa gauge.

Derived quantities:

$$R_{th} = \frac{T_{max} - T_{in}}{Q} \qquad
  P_{pump} = \Delta p \cdot V \cdot A_{inlet}$$

`A_inlet` uses the **full** domain cross-section, so the pumping power refers to
the whole heat sink rather than the simulated half.

In [ ]:
# --- case constants ----------------------------------------------------------
Q_TOTAL = 20.0                 # W, total dissipated power
T_INLET = 300.0                # K
W_DOMAIN, H_DOMAIN = 0.260, 0.128
A_INLET = W_DOMAIN * H_DOMAIN  # m^2, full-model inlet area

# --- Fluent output -----------------------------------------------------------
df = pd.DataFrame({
    "V_m_s":  [1,       2,       3,       4,       5,       6      ],
    "Tmax_K": [335.16,  322.86,  317.45,  314.51,  312.83,  311.73 ],
    "dP_Pa":  [0.252,   0.772,   1.490,   2.390,   3.480,   4.737  ],
})

df["dT_K"]      = df.Tmax_K - T_INLET
df["R_th"]      = df.dT_K / Q_TOTAL                 # K/W
df["Q_flow"]    = df.V_m_s * A_INLET                # m^3/s
df["P_pump_W"]  = df.dP_Pa * df.Q_flow              # W

# Marginal view: what each extra 1 m/s actually buys, and what it costs.
df["dR_step"]   = df.R_th.diff()
df["dP_step"]   = df.P_pump_W.diff()

df.round(4)

In [ ]:
# --- export the results table ------------------------------------------------
csv = OUT / "velocity_sweep_results.csv"
df.to_csv(csv, index=False, float_format="%.5f")
files.download(str(csv))
print(f"saved  {csv}")

## 3 · Figure 1 — Thermal resistance

The headline result: cooling improves steeply at low speed and then flattens.
Each successive 1 m/s buys roughly half of what the previous one did.

In [ ]:
fig, ax = plt.subplots()

series(ax, df.V_m_s, df.R_th, BLUE)
ax.fill_between(df.V_m_s, df.R_th, df.R_th.min()*0.55,
                color=BLUE, alpha=0.07, zorder=1)

lo, hi = df.iloc[0], df.iloc[-1]
tag(ax, lo.V_m_s, lo.R_th, f"{lo.R_th:.2f} K/W", dx=10, dy=2, color=BLUE)
tag(ax, hi.V_m_s, hi.R_th, f"{hi.R_th:.2f} K/W", dx=-16, dy=-22, color=BLUE)

# The power-law exponent is the one number a reviewer will look for.
b, a = np.polyfit(np.log(df.V_m_s), np.log(df.R_th), 1)
ax.text(0.97, 0.93, f"$R_{{th}} \\propto V^{{{b:.2f}}}$", transform=ax.transAxes,
        ha="right", va="top", fontsize=11.5, color=INK_2,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=PAPER, edgecolor=FRAME))

ax.set_xlabel("Inlet velocity   $V$   (m/s)")
ax.set_ylabel("Thermal resistance   $R_{th}$   (K/W)")
titled(ax, "Cooling improves fast, then barely at all",
       f"$R_{{th}} = (T_{{max}} - T_{{in}})/Q$    Q = {Q_TOTAL:.0f} W,  $T_{{in}}$ = {T_INLET:.0f} K")
ax.set_ylim(df.R_th.min()*0.55, df.R_th.max()*1.10)
polish(ax)

export(fig, "fig01_thermal_resistance")

## 4 · Figure 2 — Pressure drop

The hydraulic side of the trade-off. A quadratic reference curve is overlaid:
if the CFD points sit on it, the losses are inertia-dominated, which is what a
short finned array in this Reynolds range should show.

In [ ]:
fig, ax = plt.subplots()

k = np.polyfit(df.V_m_s**2, df.dP_Pa, 1)[0]
vv = np.linspace(df.V_m_s.min(), df.V_m_s.max(), 200)
ax.plot(vv, k*vv**2, color=INK_3, lw=1.5, ls=(0, (5, 3)), zorder=2,
        label=f"quadratic reference   $\\Delta p = {k:.3f}\\,V^2$")

series(ax, df.V_m_s, df.dP_Pa, ORANGE, label="CFD")

hi = df.iloc[-1]
tag(ax, hi.V_m_s, hi.dP_Pa, f"{hi.dP_Pa:.2f} Pa", dx=-14, dy=-24, color=ORANGE)

r2 = 1 - ((df.dP_Pa - k*df.V_m_s**2)**2).sum() / ((df.dP_Pa - df.dP_Pa.mean())**2).sum()
ax.text(0.97, 0.12, f"$R^2 = {r2:.4f}$", transform=ax.transAxes, ha="right",
        fontsize=10.5, color=INK_3)

ax.set_xlabel("Inlet velocity   $V$   (m/s)")
ax.set_ylabel("Pressure drop   $\\Delta p$   (Pa)")
titled(ax, "Pressure drop follows the classic $V^2$ law",
       "Area-weighted static pressure at the inlet; outlet held at 0 Pa gauge")
ax.set_ylim(0, df.dP_Pa.max()*1.12)
ax.legend(loc="upper left")
polish(ax)

export(fig, "fig02_pressure_drop")

## 5 · Figure 3 — Pumping power

Pressure drop rises with $V^2$, but the *power* to sustain the flow rises with
$V^3$, because the fan must also move proportionally more air. This cubic term
is the whole reason an optimum exists.

In [ ]:
fig, ax = plt.subplots()

k3 = np.polyfit(df.V_m_s**3, df.P_pump_W, 1)[0]
vv = np.linspace(df.V_m_s.min(), df.V_m_s.max(), 200)
ax.plot(vv, k3*vv**3, color=INK_3, lw=1.5, ls=(0, (5, 3)), zorder=2,
        label=f"cubic reference   $P = {k3:.4f}\\,V^3$")

series(ax, df.V_m_s, df.P_pump_W, ORANGE, label="CFD")
ax.fill_between(df.V_m_s, 0, df.P_pump_W, color=ORANGE, alpha=0.07, zorder=1)

hi = df.iloc[-1]
tag(ax, hi.V_m_s, hi.P_pump_W, f"{hi.P_pump_W:.2f} W", dx=-16, dy=-24, color=ORANGE)
tag(ax, 1, df.P_pump_W.iloc[0], f"{df.P_pump_W.iloc[0]*1000:.0f} mW",
    dx=10, dy=2, color=ORANGE)

ax.set_xlabel("Inlet velocity   $V$   (m/s)")
ax.set_ylabel("Pumping power   $P_{pump}$   (W)")
titled(ax, "The cost of air grows with the cube of speed",
       "$P_{pump} = \\Delta p \\cdot V \\cdot A_{inlet}$ — a 6x speed increase costs 113x the power")
ax.set_ylim(0, df.P_pump_W.max()*1.12)
ax.legend(loc="upper left")
polish(ax)

export(fig, "fig03_pumping_power")

## 6 · Figure 4 — The trade-off front

Thermal resistance against the power spent to achieve it. Every point is a
feasible operating condition; down-and-left is better. The curve turns sharply,
and past the knee each additional watt of fan power buys almost nothing.

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.2))

ax.plot(df.P_pump_W, df.R_th, color=BLUE, lw=2.2, zorder=2)
ax.scatter(df.P_pump_W, df.R_th, s=70, color=BLUE,
           edgecolor=PANEL, linewidth=1.8, zorder=3)

for _, r in df.iterrows():
    ax.annotate(f"{r.V_m_s:.0f} m/s", (r.P_pump_W, r.R_th),
                textcoords="offset points", xytext=(11, 7),
                fontsize=10, color=INK_2, fontweight="bold")

# Quantify the diminishing return between the two extreme steps.
first = df.iloc[1]; last = df.iloc[-1]; prev = df.iloc[-2]
gain_lo = (df.R_th.iloc[0] - first.R_th) / (first.P_pump_W - df.P_pump_W.iloc[0])
gain_hi = (prev.R_th - last.R_th) / (last.P_pump_W - prev.P_pump_W)
ax.text(0.97, 0.90,
        f"1 → 2 m/s:   {gain_lo:7.2f} K/W per extra watt\n"
        f"5 → 6 m/s:   {gain_hi:7.2f} K/W per extra watt",
        transform=ax.transAxes, ha="right", va="top", fontsize=10.5,
        color=INK_2, family="DejaVu Sans Mono",
        bbox=dict(boxstyle="round,pad=0.6", facecolor=PAPER, edgecolor=FRAME))

ax.set_xlabel("Pumping power   $P_{pump}$   (W)")
ax.set_ylabel("Thermal resistance   $R_{th}$   (K/W)")
titled(ax, "Buying the last few degrees costs the most power",
       "Each marker is one operating point — down-and-left is better")
ax.set_xlim(-0.03, df.P_pump_W.max()*1.20)
ax.set_ylim(0, df.R_th.max()*1.12)
polish(ax)

export(fig, "fig04_tradeoff_front")

## 7 · Figure 5 — Combined objective and the optimum

Both quantities are normalised against the mid-sweep case so they can be added:

$$J(V) = \frac{R_{th}(V)}{R_{th,ref}} + w\,\frac{P_{pump}(V)}{P_{pump,ref}}$$

The weight $w$ is an explicit engineering judgement, not a physical constant:
$w = 1$ treats a 1 % thermal gain and a 1 % power increase as equally
important. Three weights are shown so the sensitivity of the answer to that
judgement is visible rather than hidden.

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.2))

ref = df.iloc[len(df)//2]                      # normalise at V = 4 m/s
vv = np.linspace(df.V_m_s.min(), df.V_m_s.max(), 600)

WEIGHTS = [(0.5, TEAL, "-"), (1.0, CRIMSON, "-"), (2.0, BLUE, "--")]
optima = []

for w, colour, ls in WEIGHTS:
    j = df.R_th/ref.R_th + w*df.P_pump_W/ref.P_pump_W
    jj = np.polyval(np.polyfit(df.V_m_s, j, 3), vv)
    v_opt = vv[jj.argmin()]
    optima.append((w, v_opt, jj.min()))

    lw = 2.6 if w == 1.0 else 1.8
    ax.plot(vv, jj, color=colour, lw=lw, ls=ls, zorder=3 if w == 1.0 else 2,
            label=f"$w = {w:g}$   →   $V_{{opt}} = {v_opt:.2f}$ m/s")
    ax.scatter([v_opt], [jj.min()], s=55, color=colour,
               edgecolor=PANEL, linewidth=1.8, zorder=4)
    if w == 1.0:
        ax.scatter(df.V_m_s, j, s=42, color=colour,
                   edgecolor=PANEL, linewidth=1.6, zorder=4)

v_star = [o[1] for o in optima if o[0] == 1.0][0]
ax.axvline(v_star, color=INK_3, lw=1.1, ls=(0, (3, 3)), zorder=1)

ax.set_xlabel("Inlet velocity   $V$   (m/s)")
ax.set_ylabel("Combined objective   $J$   (dimensionless)")
titled(ax, f"The balanced optimum sits at {v_star:.1f} m/s",
       f"$J = R/R_{{ref}} + w \\cdot P/P_{{ref}}$, normalised at V = {ref.V_m_s:.0f} m/s")
ax.legend(loc="upper center", ncol=1)
polish(ax)

export(fig, "fig05_combined_objective")

print("\nSensitivity of the optimum to the weighting:")
for w, v_opt, _ in optima:
    print(f"  w = {w:4.1f}  ->  V_opt = {v_opt:.2f} m/s")

## 8 · Validation

Two independent checks.

**Energy balance (from Fluent).** Heat into the heated patch 10.000001 W over
the half-model; net imbalance across all external boundaries 2.3 x 10⁻⁵ W —
an error of 0.0002 %. That proves the solver conserved energy. It does *not*
prove the physics is right.

**Analytical fin-array model.** The check that could actually catch a wrong
answer. The heat sink sits in an open domain, not a duct: the channels are
short ($L/D_h \approx 6$) and much of the air bypasses the array, so the fin
surfaces behave like flat plates growing fresh boundary layers rather than a
fully-developed channel. The reference correlation is therefore

$$\mathrm{Nu}_L = 0.664\,\mathrm{Re}_L^{1/2}\,\mathrm{Pr}^{1/3}$$

with fin efficiency from the standard straight-fin solution. The fully-developed
parallel-plate result ($\mathrm{Nu} = 7.54$) is computed alongside it — not as a
competing answer, but to show how badly a ducted-flow assumption would
mis-predict the velocity trend.

In [ ]:
# --- geometry and air properties (Fluent's constant-property air) ------------
L_FIN, H_FIN, T_FIN, N_FIN, PITCH = 0.060, 0.025, 0.0015, 10, 0.0062
L_BASE = W_BASE = 0.060
T_BASE, K_AL = 0.003, 202.0
GAP = PITCH - T_FIN

RHO, MU, K_AIR, CP = 1.225, 1.7894e-5, 0.0242, 1006.43
PR = MU * CP / K_AIR

A_FIN  = N_FIN * 2 * L_FIN * H_FIN + N_FIN * L_FIN * T_FIN   # sides + tips
A_BASE = L_BASE * W_BASE - N_FIN * L_FIN * T_FIN             # exposed base
R_COND = T_BASE / (K_AL * L_BASE * W_BASE)


def fin_efficiency(h):
    lc = H_FIN + T_FIN / 2
    m = np.sqrt(2 * h / (K_AL * T_FIN))
    return np.tanh(m * lc) / (m * lc)


def resistance(h):
    eta = fin_efficiency(h)
    return 1 / (h * (eta * A_FIN + A_BASE)) + R_COND, eta


rows = []
for v, r_cfd in zip(df.V_m_s, df.R_th):
    re_l = RHO * v * L_FIN / MU
    h_fp = 0.664 * re_l**0.5 * PR**(1/3) * K_AIR / L_FIN     # flat plate
    h_du = 7.54 * K_AIR / (2 * GAP)                          # ducted, for contrast
    r_fp, eta = resistance(h_fp)
    r_du, _   = resistance(h_du)
    rows.append(dict(V_m_s=v, Re_L=re_l, h_W_m2K=h_fp, eta_fin=eta,
                     R_analytical=r_fp, R_ducted=r_du, R_CFD=r_cfd,
                     deviation_pct=100*(r_cfd - r_fp)/r_fp))

val = pd.DataFrame(rows)

exp_cfd = np.polyfit(np.log(val.V_m_s), np.log(val.R_CFD), 1)[0]
exp_fp  = np.polyfit(np.log(val.V_m_s), np.log(val.R_analytical), 1)[0]
exp_du  = np.polyfit(np.log(val.V_m_s), np.log(val.R_ducted), 1)[0]

print(f"Pr = {PR:.3f}    wetted area = {(A_FIN + A_BASE)*1e4:.0f} cm²    "
      f"R_cond(base) = {R_COND*1e3:.2f} mK/W\n")
print(f"Velocity scaling:   CFD  R ∝ V^{exp_cfd:+.2f}")
print(f"                    flat-plate model  R ∝ V^{exp_fp:+.2f}")
print(f"                    ducted model      R ∝ V^{exp_du:+.2f}   <- no velocity "
      f"sensitivity at all: the wrong model\n")
print(f"Deviation band: {val.deviation_pct.min():.0f} % to "
      f"{val.deviation_pct.max():.0f} %, consistent sign\n")

val.round(3)

### Validation table (figure)

Rendered as a figure so it can be dropped straight into the report, and
exported as CSV for the repository.

In [ ]:
fig, ax = plt.subplots(figsize=(9.6, 3.5))
ax.set_facecolor(PAPER); ax.axis("off")

cols = ["V\n(m/s)", "$Re_L$", "h\n(W/m²K)", "$\\eta_{fin}$",
        "$R_{analytical}$\n(K/W)", "$R_{CFD}$\n(K/W)", "Deviation\n(%)"]
body = [[f"{r.V_m_s:.0f}", f"{r.Re_L:,.0f}", f"{r.h_W_m2K:.1f}", f"{r.eta_fin:.3f}",
         f"{r.R_analytical:.3f}", f"{r.R_CFD:.3f}", f"{r.deviation_pct:+.1f}"]
        for _, r in val.iterrows()]

# bbox leaves the top strip free for the title block
tbl = ax.table(cellText=body, colLabels=cols, cellLoc="center",
               bbox=[0, 0, 1, 0.84])
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(FRAME); cell.set_linewidth(0.7)
    if row == 0:                                  # header
        cell.set_facecolor("#2b2a26"); cell.set_height(0.16)
        cell.set_text_props(color="#ffffff", fontweight="bold")
    else:
        cell.set_facecolor(PANEL if row % 2 else PAPER)
        if col == 6:                              # deviation column carries the verdict
            cell.set_text_props(color=CRIMSON, fontweight="bold")
        elif col == 5:
            cell.set_text_props(color=BLUE, fontweight="bold")

ax.text(0, 1.10, "Analytical validation of the CFD thermal resistance",
        transform=ax.transAxes, fontsize=13.5, fontweight="bold",
        family="DejaVu Serif", color=INK, va="bottom")
ax.text(0, 0.94, "Laminar flat-plate correlation with straight-fin efficiency · "
        "CFD consistently predicts better cooling, as expected",
        transform=ax.transAxes, fontsize=10, color=INK_3, va="bottom")

export(fig, "fig06_validation_table")

csv = OUT / "validation.csv"
val.to_csv(csv, index=False, float_format="%.5f")
files.download(str(csv))
print(f"saved  {csv}")

## 9 · Figure 7 — Validation, plotted

Two curves that should sit close and fall at a similar rate, plus the ducted
model as a cautionary flat line.

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.0))

series(ax, val.V_m_s, val.R_CFD, BLUE, label="CFD (Fluent, k-$\\omega$ SST)")
series(ax, val.V_m_s, val.R_analytical, ORANGE, marker="s", ls="--",
       label="Analytical — flat-plate + fin efficiency")
ax.plot(val.V_m_s, val.R_ducted, color=INK_3, lw=1.5, ls=(0, (2, 3)),
        label="Analytical — fully-developed duct (inappropriate here)")

ax.fill_between(val.V_m_s, val.R_CFD, val.R_analytical,
                color=BLUE, alpha=0.09, zorder=1)

mid = len(val)//2
ax.annotate(f"{val.deviation_pct.iloc[mid]:.0f} %",
            (val.V_m_s.iloc[mid], (val.R_CFD.iloc[mid] + val.R_analytical.iloc[mid])/2),
            textcoords="offset points", xytext=(12, -4),
            fontsize=10.5, color=INK_2, fontweight="bold")

ax.set_xlabel("Inlet velocity   $V$   (m/s)")
ax.set_ylabel("Thermal resistance   $R_{th}$   (K/W)")
titled(ax, "CFD and a hand calculation agree within 29 %",
       "Same magnitude, same trend, consistent sign — the CFD result is credible")
ax.set_ylim(0, max(val.R_analytical.max(), val.R_ducted.max())*1.12)
ax.legend(loc="upper right")
polish(ax)

export(fig, "fig07_validation_curves")

## 10 · Summary panel

A single figure for the front of the report or the repository README: the
thermal side, the hydraulic side, and the optimum that falls out of the two.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))

# -- panel A: thermal ---------------------------------------------------------
a = axes[0]
series(a, df.V_m_s, df.R_th, BLUE)
a.fill_between(df.V_m_s, df.R_th, 0, color=BLUE, alpha=0.07)
a.set_xlabel("Inlet velocity  (m/s)"); a.set_ylabel("$R_{th}$  (K/W)")
a.set_title("A · Thermal resistance", loc="left", fontsize=12,
            fontweight="bold", family="DejaVu Serif", color=INK, pad=14)
a.set_ylim(0, df.R_th.max()*1.12); polish(a)

# -- panel B: hydraulic -------------------------------------------------------
b = axes[1]
series(b, df.V_m_s, df.P_pump_W, ORANGE)
b.fill_between(df.V_m_s, df.P_pump_W, 0, color=ORANGE, alpha=0.07)
b.set_xlabel("Inlet velocity  (m/s)"); b.set_ylabel("$P_{pump}$  (W)")
b.set_title("B · Pumping power", loc="left", fontsize=12,
            fontweight="bold", family="DejaVu Serif", color=INK, pad=14)
b.set_ylim(0, df.P_pump_W.max()*1.12); polish(b)

# -- panel C: the optimum -----------------------------------------------------
c = axes[2]
j = df.R_th/ref.R_th + df.P_pump_W/ref.P_pump_W
jj = np.polyval(np.polyfit(df.V_m_s, j, 3), vv)
c.plot(vv, jj, color=CRIMSON, lw=2.4)
c.scatter(df.V_m_s, j, s=45, color=CRIMSON, edgecolor=PANEL, linewidth=1.6, zorder=3)
c.axvline(v_star, color=INK_3, lw=1.1, ls=(0, (3, 3)))
c.annotate(f"$V_{{opt}}$ = {v_star:.1f} m/s", (v_star, jj.min()),
           textcoords="offset points", xytext=(-96, 26),
           fontsize=11, fontweight="bold", color=CRIMSON)
c.set_xlabel("Inlet velocity  (m/s)"); c.set_ylabel("$J$  (–)")
c.set_title("C · Combined objective", loc="left", fontsize=12,
            fontweight="bold", family="DejaVu Serif", color=INK, pad=14)
span = jj.max() - jj.min()
c.set_ylim(jj.min() - 0.20*span, jj.max() + 0.08*span); polish(c)

fig.tight_layout()                      # lay the panels out first ...
fig.subplots_adjust(top=0.84)           # ... then reserve a strip for the title
fig.text(0.007, 1.05, "Air-cooled heat sink — velocity study at Q = 20 W",
         ha="left", fontsize=15, fontweight="bold",
         family="DejaVu Serif", color=INK)
fig.text(0.007, 1.00, "Plate-fin Al-6061, 10 fins · ANSYS Fluent 2026 R1, "
         "k-$\\omega$ SST, 418k cells", ha="left", fontsize=10.5, color=INK_3)

export(fig, "fig08_summary_panel")

## 11 · Download everything

Bundles every figure and CSV into one archive — the file to commit to the
repository.

In [ ]:
import shutil
archive = shutil.make_archive("heatsink_cfd_outputs", "zip", OUT)
files.download(archive)
print("contents:")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:34s} {p.stat().st_size/1024:7.1f} KB")